In [1]:
import sys
!{sys.executable} -m pip install flask flask-sqlalchemy pyttsx3 pygame APScheduler
print("Installation klar!")


Installation klar!


In [2]:
import json
import sqlite3

# Config
config = {
  "people": [
    {"id": 1, "name": "Mamma", "color": "#FF69B4"},
    {"id": 2, "name": "Pappa", "color": "#4169E1"},
    {"id": 3, "name": "Barn 1", "color": "#90EE90"}
  ]
}
with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)
print("✅ config.json skapad")

# Databas
conn = sqlite3.connect('project_e-da.db')
conn.execute('''CREATE TABLE IF NOT EXISTS event 
                (id INTEGER PRIMARY KEY AUTOINCREMENT,
                 person_id INTEGER,
                 title TEXT,
                 event_time TEXT,
                 is_active INTEGER DEFAULT 1)''')
conn.commit()
conn.close()
print("✅ Databas skapad")


✅ config.json skapad
✅ Databas skapad


In [3]:
import sqlite3
from datetime import datetime, timedelta

# Lägg till en test-händelse (om 10 sekunder)
test_time = datetime.now() + timedelta(seconds=10)

conn = sqlite3.connect('project_e-da.db')
conn.execute("INSERT INTO event (person_id, title, event_time, is_active) VALUES (?, ?, ?, ?)",
             (1, "Test läxor", test_time.isoformat(), 1))
conn.commit()
conn.close()

print(f"✅ Händelse lagd kl {test_time.strftime('%H:%M:%S')}")


✅ Händelse lagd kl 17:23:52


In [2]:
import sqlite3
import json
import subprocess

with open('config.json') as f:
    config = json.load(f)

conn = sqlite3.connect('project_e-da.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM event WHERE is_active = 1")
row = cursor.fetchone()
conn.close()

if row:
    person_name = next(p['name'] for p in config['people'] if p['id'] == row[1])
    text = f"{person_name}, dags för {row[2]}!"
    print(f"🚨 ALARM: {text}")
    subprocess.run(['powershell', '-Command',
        f'Add-Type -AssemblyName System.Speech; '
        f'$s = New-Object System.Speech.Synthesis.SpeechSynthesizer; '
        f'$s.Speak("{text}")'])
    print("✅ Alarm OK!")
else:
    print("❌ Inga aktiva händelser")


🚨 ALARM: Mamma, dags för sdf!
✅ Alarm OK!
